In [ ]:
# pip install -U optax flax equinox chex diffrax

In [ ]:
"""
08_volterra_capacity_and_transfer_learning

Volterra-Fredholm Continuous Spectral Attention
Precision: Native bfloat16 mixed precision compilation (@jax.jit)
Dataset: WikiText-2 (Direct GitHub mirrors, GPT-2 BPE Tokenizer, 50,257 vocabulary)

Core Mathematical Components:
  1. Continuous Fredholm Integral Operator with Gauss-Chebyshev Quadrature.
  2. Causal Dissipative Semigroup: exp(-gamma_m * (t - s)) with learnable decay rates.
  3. Hyperspherical Conical Confinement on S^{d-1} (zero-softmax, zero unconstrained exponentials).
  4. Associative Content Contraction without intermediate 5D tensor generation.
  5. Calibrated Architecture: Strict separation of Backbone Parameters vs. Embedding Tables.
"""

import os
import math
import time
import urllib.request
from typing import Any, Dict, Optional, Tuple

import numpy as np

# Configure XLA memory allocation
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import jax
import jax.numpy as jnp
from jax import random
import flax.linen as nn
from flax.training import train_state
import optax

# -----------------------------------------------------------------------------
# 0. Global Configuration and Hardware Setup
# -----------------------------------------------------------------------------
DEVICES = jax.devices()
BLOCK_SIZE: int = 512
BATCH_SIZE: int = 32
TOTAL_STEPS: int = 1200
EVAL_INTERVAL: int = 150
EVAL_ITERS: int = 40
LEARNING_RATE: float = 6e-4
WEIGHT_DECAY: float = 0.1
SEED: int = 42
DTYPE: jnp.dtype = jnp.bfloat16

# Architectural dimensions (Default: Calibrated Medium Scale)
D_MODEL: int = 256
N_LAYERS: int = 4
N_HEADS: int = 4
M_MODES: int = 16


# -----------------------------------------------------------------------------
# 1. Dataset Ingestion Pipeline
# -----------------------------------------------------------------------------
def load_wikitext2_corpus() -> Tuple[np.ndarray, np.ndarray, int]:
    """
    Downloads WikiText-2 directly from official repositories.
    Encodes raw text into token IDs via GPT-2 BPE.
    """
    data_dir = "/tmp/wikitext2_dataset"
    os.makedirs(data_dir, exist_ok=True)
    train_path = os.path.join(data_dir, "train.txt")
    valid_path = os.path.join(data_dir, "valid.txt")

    urls = {
        "train": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt",
        "valid": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/valid.txt"
    }
    headers = {"User-Agent": "Mozilla/5.0"}

    for split, url in urls.items():
        dest = train_path if split == "train" else valid_path
        if not os.path.exists(dest) or os.path.getsize(dest) < 1000:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=30) as resp, open(dest, "wb") as f:
                f.write(resp.read())

    with open(train_path, "r", encoding="utf-8") as f:
        train_text = f.read()
    with open(valid_path, "r", encoding="utf-8") as f:
        valid_text = f.read()

    try:
        import tiktoken
        tokenizer = tiktoken.get_encoding("gpt2")
        train_tokens = tokenizer.encode_ordinary(train_text)
        valid_tokens = tokenizer.encode_ordinary(valid_text)
        vocab_size = tokenizer.n_vocab
    except ImportError:
        train_tokens = list(train_text.encode("utf-8"))
        valid_tokens = list(valid_text.encode("utf-8"))
        vocab_size = 256

    train_array = np.array(train_tokens, dtype=np.int32)
    valid_array = np.array(valid_tokens, dtype=np.int32)
    return train_array, valid_array, vocab_size


TRAIN_DATA, VAL_DATA, VOCAB_SIZE = load_wikitext2_corpus()


def sample_batch(data_array: np.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
    max_start = len(data_array) - BLOCK_SIZE - 1
    indices = np.random.randint(0, max_start, size=(BATCH_SIZE,))
    x = np.stack([data_array[i : i + BLOCK_SIZE] for i in indices])
    y = np.stack([data_array[i + 1 : i + BLOCK_SIZE + 1] for i in indices])
    return jnp.array(x, dtype=jnp.int32), jnp.array(y, dtype=jnp.int32)


# -----------------------------------------------------------------------------
# 2. Mathematical Primitives & Geometric Normalization
# -----------------------------------------------------------------------------
def project_conical(x: jnp.ndarray, eps: float = 1e-7) -> jnp.ndarray:
    """
    Geodesic projection onto the unit hypersphere shell S^{d-1}:
        proj(x) = sqrt(d) * (x / ||x||_2)
    """
    scale = jnp.sqrt(float(x.shape[-1]))
    norm = jnp.linalg.norm(x.astype(jnp.float32), axis=-1, keepdims=True) + eps
    return (scale * (x.astype(jnp.float32) / norm)).astype(x.dtype)


class ConicalNorm(nn.Module):
    dim: int
    eps: float = 1e-7

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        return project_conical(x, self.eps)


def identity_matrix_init(rng: Any, shape: Tuple[int, int], dtype: jnp.dtype = jnp.float32) -> jnp.ndarray:
    return jnp.eye(shape[0], shape[1], dtype=dtype)


# -----------------------------------------------------------------------------
# 3. Attention Operators
# -----------------------------------------------------------------------------
class ScaledVolterraAttention(nn.Module):
    """
    Continuous Volterra-Fredholm Spectral Attention.
    Discretizes the causal integral via Gauss-Chebyshev nodes and dynamic semigroups.
    """
    d_model: int
    n_heads: int
    m_modes: int
    block_size: int = BLOCK_SIZE
    dtype: jnp.dtype = DTYPE

    def setup(self) -> None:
        self.head_dim = self.d_model // self.n_heads

        # First-kind Chebyshev polynomial base T_m(x)
        t = jnp.arange(self.block_size, dtype=jnp.float32)
        m = jnp.arange(self.m_modes, dtype=jnp.float32)[None, :]
        t_normalized = (2.0 * t[:, None] / max(self.block_size - 1, 1)) - 1.0
        t_clipped = jnp.minimum(jnp.maximum(t_normalized, -1.0 + 1e-6), 1.0 - 1e-6)
        cheb = jnp.cos(m * jnp.arccos(t_clipped))

        # Analytic Gauss-Chebyshev density weights
        x_s = (2.0 * t + 1.0) / (2.0 * self.block_size)
        w_cheb = 1.0 / jnp.sqrt(x_s * (1.0 - x_s) + 1e-5)
        w_cheb = (w_cheb / jnp.mean(w_cheb))[:, None]

        self.base_phi = cheb
        self.base_psi = cheb * w_cheb

        # Causal temporal distance grid
        t_grid = t[:, None]
        s_grid = t[None, :]
        self.delta_ts = jnp.maximum(t_grid - s_grid, 0.0)
        self.causal_mask = jnp.tril(jnp.ones((self.block_size, self.block_size), dtype=jnp.bool_))

    @nn.compact
    def __call__(self, x: jnp.ndarray, audit: bool = False) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        b, t, c = x.shape

        c_attn = nn.Dense(3 * self.d_model, use_bias=False, dtype=self.dtype, name="c_attn")
        c_proj = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="c_proj")

        # Modal coupling eigenvalues
        lambdas = self.param(
            "lambdas",
            lambda rng: jnp.ones((self.n_heads, self.m_modes), dtype=jnp.float32) / math.sqrt(self.m_modes)
        )
        init_gammas = jnp.broadcast_to(
            jnp.linspace(math.log(0.005), math.log(0.20), self.m_modes),
            (self.n_heads, self.m_modes)
        )
        log_gamma = self.param("log_gamma", lambda rng: init_gammas)

        adapt_phi = nn.Dense(self.m_modes, use_bias=False, kernel_init=identity_matrix_init, name="adapt_phi")
        adapt_psi = nn.Dense(self.m_modes, use_bias=False, kernel_init=identity_matrix_init, name="adapt_psi")

        qkv = c_attn(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        q = q.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)
        k = k.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)
        v = v.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)

        q_proj = project_conical(q)
        k_proj = project_conical(k)

        phi = adapt_phi(self.base_phi[:t, :])
        psi = adapt_psi(self.base_psi[:t, :])

        # Dissipative semigroup decay: exp(-gamma * (t - s))
        gamma = jnp.minimum(jnp.maximum(jnp.exp(log_gamma), 1e-4), 2.0)
        gamma_exp = gamma[:, :, None, None]
        delta = self.delta_ts[:t, :t][None, None, :, :]

        decay_matrix = jnp.exp(-gamma_exp * delta)
        base_kernel = jnp.einsum("tm,sm->mst", phi, psi)[None, :, :, :]

        lam = lambdas[:, :, None, None]
        causal_kernel = jnp.sum(lam * base_kernel * decay_matrix, axis=1)
        causal_kernel = jnp.where(self.causal_mask[:t, :t][None, :, :], causal_kernel, 0.0)

        # Associative state integration
        input_content = k_proj * v
        integrated_state = jnp.matmul(causal_kernel.astype(self.dtype)[None, :, :, :], input_content)
        state_conical = project_conical(integrated_state)

        out = q_proj * state_conical
        out = out.swapaxes(1, 2).reshape((b, t, c))

        if audit:
            max_val = jnp.max(jnp.abs(causal_kernel))
            k_abs = jnp.abs(causal_kernel)
            probs = k_abs / (jnp.sum(k_abs, axis=-1, keepdims=True) + 1e-7)
            probs = jnp.maximum(probs, 1e-9)
            entropy = -jnp.mean(jnp.sum(probs * jnp.log(probs), axis=-1))
        else:
            max_val = jnp.array(0.0, dtype=jnp.float32)
            entropy = jnp.array(0.0, dtype=jnp.float32)

        return c_proj(out), max_val, entropy


class StandardCausalAttention(nn.Module):
    """Canonical causal multi-head self-attention with Softmax (GPT-2 baseline)."""
    d_model: int
    n_heads: int
    block_size: int = BLOCK_SIZE
    dtype: jnp.dtype = DTYPE

    def setup(self) -> None:
        self.head_dim = self.d_model // self.n_heads
        self.causal_mask = jnp.tril(jnp.ones((self.block_size, self.block_size), dtype=jnp.bool_))

    @nn.compact
    def __call__(self, x: jnp.ndarray, audit: bool = False) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        b, t, c = x.shape
        c_attn = nn.Dense(3 * self.d_model, use_bias=False, dtype=self.dtype, name="c_attn")
        c_proj = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="c_proj")

        qkv = c_attn(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        q = q.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)
        k = k.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)
        v = v.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)

        scale = 1.0 / math.sqrt(self.head_dim)
        scores = jnp.matmul(q.astype(jnp.float32), k.astype(jnp.float32).swapaxes(-2, -1)) * scale
        scores = jnp.where(self.causal_mask[:t, :t][None, None, :, :], scores, -1e9)
        attn_weights = jax.nn.softmax(scores, axis=-1).astype(self.dtype)

        y = jnp.matmul(attn_weights, v)
        y = y.swapaxes(1, 2).reshape((b, t, c))

        if audit:
            max_val = jnp.max(scores)
            p = jnp.maximum(attn_weights.astype(jnp.float32), 1e-9)
            entropy = -jnp.mean(jnp.sum(p * jnp.log(p), axis=-1))
        else:
            max_val = jnp.array(0.0, dtype=jnp.float32)
            entropy = jnp.array(0.0, dtype=jnp.float32)

        return c_proj(y), max_val, entropy


# -----------------------------------------------------------------------------
# 4. Modular Transformer Blocks & Architecture
# -----------------------------------------------------------------------------
class VolterraBlock(nn.Module):
    d_model: int
    n_heads: int
    m_modes: int
    block_size: int = BLOCK_SIZE
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(self, x: jnp.ndarray, audit: bool = False) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        norm1 = ConicalNorm(self.d_model, name="norm1")
        norm2 = ConicalNorm(self.d_model, name="norm2")
        attn = ScaledVolterraAttention(
            d_model=self.d_model,
            n_heads=self.n_heads,
            m_modes=self.m_modes,
            block_size=self.block_size,
            dtype=self.dtype,
            name="attn"
        )
        attn_out, max_val, entropy = attn(norm1(x), audit=audit)
        x = x + attn_out

        mlp_fc1 = nn.Dense(4 * self.d_model, use_bias=False, dtype=self.dtype, name="mlp_fc1")
        mlp_fc2 = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="mlp_fc2")
        x = x + mlp_fc2(nn.silu(mlp_fc1(norm2(x))))
        return x, max_val, entropy


class StandardBlock(nn.Module):
    d_model: int
    n_heads: int
    block_size: int = BLOCK_SIZE
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(self, x: jnp.ndarray, audit: bool = False) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        norm1 = nn.LayerNorm(name="norm1")
        norm2 = nn.LayerNorm(name="norm2")
        attn = StandardCausalAttention(
            d_model=self.d_model,
            n_heads=self.n_heads,
            block_size=self.block_size,
            dtype=self.dtype,
            name="attn"
        )
        attn_out, max_val, entropy = attn(norm1(x), audit=audit)
        x = x + attn_out

        mlp_fc1 = nn.Dense(4 * self.d_model, use_bias=False, dtype=self.dtype, name="mlp_fc1")
        mlp_fc2 = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="mlp_fc2")
        x = x + mlp_fc2(jax.nn.gelu(mlp_fc1(norm2(x))))
        return x, max_val, entropy


class AutoregressiveLM(nn.Module):
    vocab_size: int = VOCAB_SIZE
    d_model: int = D_MODEL
    n_heads: int = N_HEADS
    n_layers: int = N_LAYERS
    m_modes: int = M_MODES
    attention_type: str = "volterra"
    block_size: int = BLOCK_SIZE
    dtype: jnp.dtype = DTYPE

    @nn.compact
    def __call__(
        self,
        idx: jnp.ndarray,
        targets: Optional[jnp.ndarray] = None,
        audit: bool = False
    ) -> Tuple[jnp.ndarray, Any, jnp.ndarray, jnp.ndarray]:
        b, t = idx.shape

        tok_embed = self.param(
            "token_emb",
            nn.initializers.normal(stddev=0.02),
            (self.vocab_size, self.d_model),
            self.dtype
        )
        pos_embed = self.param(
            "pos_emb",
            nn.initializers.normal(stddev=0.02),
            (self.block_size, self.d_model),
            self.dtype
        )

        pos = jnp.arange(t)[None, :]
        x = tok_embed[idx] + pos_embed[pos]

        max_vals, entropies = [], []
        for i in range(self.n_layers):
            if self.attention_type == "volterra":
                x, mv, ent = VolterraBlock(
                    d_model=self.d_model,
                    n_heads=self.n_heads,
                    m_modes=self.m_modes,
                    block_size=self.block_size,
                    dtype=self.dtype,
                    name=f"block_{i}"
                )(x, audit=audit)
            else:
                x, mv, ent = StandardBlock(
                    d_model=self.d_model,
                    n_heads=self.n_heads,
                    block_size=self.block_size,
                    dtype=self.dtype,
                    name=f"block_{i}"
                )(x, audit=audit)

            if audit:
                max_vals.append(mv)
                entropies.append(ent)

        if self.attention_type == "volterra":
            x = ConicalNorm(self.d_model, name="norm_final")(x)
        else:
            x = nn.LayerNorm(name="norm_final")(x)

        # Weight-tied linear prediction head
        logits = jnp.matmul(x, tok_embed.T)

        loss = None
        if targets is not None:
            loss = optax.softmax_cross_entropy_with_integer_labels(
                logits=logits.astype(jnp.float32),
                labels=targets
            ).mean()

        if audit:
            peak_val = jnp.max(jnp.stack(max_vals))
            avg_ent = jnp.mean(jnp.stack(entropies))
        else:
            peak_val = jnp.array(0.0, dtype=jnp.float32)
            avg_ent = jnp.array(0.0, dtype=jnp.float32)

        return logits, loss, peak_val, avg_ent


# -----------------------------------------------------------------------------
# 5. Training State and Compiled Step Functions
# -----------------------------------------------------------------------------
def build_training_state(rng: Any, model: nn.Module, lr: float) -> Tuple[train_state.TrainState, int, int]:
    dummy_x = jnp.ones((BATCH_SIZE, BLOCK_SIZE), dtype=jnp.int32)
    params = model.init(rng, dummy_x, targets=dummy_x)["params"]

    total_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
    embed_params = VOCAB_SIZE * model.d_model + BLOCK_SIZE * model.d_model
    backbone_params = total_params - embed_params

    schedule = optax.cosine_decay_schedule(init_value=lr, decay_steps=TOTAL_STEPS)
    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=schedule, weight_decay=WEIGHT_DECAY, b1=0.9, b2=0.95)
    )
    state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)
    return state, total_params, backbone_params


@jax.jit
def train_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[train_state.TrainState, jnp.ndarray]:
    def loss_fn(params):
        _, loss, _, _ = state.apply_fn({"params": params}, x, targets=y, audit=False)
        return loss

    grad_fn = jax.value_and_grad(loss_fn)
    loss, grads = grad_fn(state.params)
    new_state = state.apply_gradients(grads=grads)
    return new_state, loss


@jax.jit
def eval_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    _, loss, max_val, entropy = state.apply_fn({"params": state.params}, x, targets=y, audit=True)
    return loss, max_val, entropy


def evaluate(state: train_state.TrainState, iters: int = EVAL_ITERS) -> Tuple[float, float, float, float]:
    losses, max_vals, entropies = [], [], []
    for _ in range(iters):
        xb, yb = sample_batch(VAL_DATA)
        loss, mv, ent = eval_step(state, xb, yb)
        losses.append(float(loss))
        max_vals.append(float(mv))
        entropies.append(float(ent))

    val_loss = float(np.mean(losses))
    ppl = math.exp(min(val_loss, 15.0))
    peak_val = float(np.max(max_vals))
    mean_ent = float(np.mean(entropies))
    return val_loss, ppl, peak_val, mean_ent


# -----------------------------------------------------------------------------
# 6. Benchmark Execution Loop
# -----------------------------------------------------------------------------
def run_benchmark() -> None:
    models_to_evaluate = [
        ("Volterra-Fundamental (M=16)", "volterra", M_MODES),
        ("Reference Baseline (Softmax GPT-2)", "softmax", 0),
    ]

    print("=" * 130)
    print(f"[INFO] Device: {DEVICES[0].device_kind.upper()} | Backend: {jax.default_backend()} | Precision: bfloat16")
    print(f"[INFO] Dataset: WikiText-2 | Tokens: Train={len(TRAIN_DATA):,} | Val={len(VAL_DATA):,} | Vocab={VOCAB_SIZE}")
    print(f"[INFO] Protocol: Steps={TOTAL_STEPS} | Batch={BATCH_SIZE} | Context={BLOCK_SIZE} | D_Model={D_MODEL}")
    print("=" * 130)

    summary_records = []

    for name, attn_type, m_val in models_to_evaluate:
        rng = random.PRNGKey(SEED)
        rng, init_rng = random.split(rng)

        model = AutoregressiveLM(
            vocab_size=VOCAB_SIZE,
            d_model=D_MODEL,
            n_heads=N_HEADS,
            n_layers=N_LAYERS,
            m_modes=m_val,
            attention_type=attn_type,
            block_size=BLOCK_SIZE,
            dtype=DTYPE,
        )

        state, total_params, backbone_params = build_training_state(init_rng, model, LEARNING_RATE)
        tokens_per_param = len(TRAIN_DATA) / max(backbone_params, 1)

        print(f"\n[INFO] Starting Architecture: {name}")
        print(f"       Total Parameters: {total_params / 1e6:.2f}M | Backbone: {backbone_params / 1e3:.1f}k | Tokens/Param: {tokens_per_param:.1f}:1")

        # Compile XLA graphs
        t_compile = time.time()
        dummy_xb, dummy_yb = sample_batch(TRAIN_DATA)
        state, _ = train_step(state, dummy_xb, dummy_yb)
        _ = eval_step(state, dummy_xb, dummy_yb)
        jax.block_until_ready(state.params)
        print(f"       XLA compilation completed in {time.time() - t_compile:.2f}s")

        t0 = time.time()
        best_loss = float("inf")
        best_step = 0

        for step in range(1, TOTAL_STEPS + 1):
            xb, yb = sample_batch(TRAIN_DATA)
            state, loss = train_step(state, xb, yb)

            if step % EVAL_INTERVAL == 0 or step == TOTAL_STEPS:
                jax.block_until_ready(loss)
                v_loss, v_ppl, v_val, v_ent = evaluate(state)
                if v_loss < best_loss:
                    best_loss = v_loss
                    best_step = step

                throughput = (step * BATCH_SIZE * BLOCK_SIZE) / (time.time() - t0)
                print(
                    f"  Step {step:>4}/{TOTAL_STEPS} | Val Loss: {v_loss:.4f} | PPL: {v_ppl:>6.2f} | "
                    f"Entropy: {v_ent:.3f} | Kernel Max: {v_val:>5.1f} | Throughput: {throughput / 1e3:>5.1f}k tok/s"
                )

        elapsed = time.time() - t0
        final_loss, final_ppl, final_val, final_ent = evaluate(state, iters=80)
        final_throughput = (TOTAL_STEPS * BATCH_SIZE * BLOCK_SIZE) / elapsed

        summary_records.append({
            "name": name,
            "backbone": f"{backbone_params / 1e3:.0f}k",
            "total": f"{total_params / 1e6:.1f}M",
            "ratio": f"{tokens_per_param:.1f}:1",
            "best_loss": best_loss,
            "best_step": best_step,
            "final_ppl": final_ppl,
            "entropy": final_ent,
            "throughput": final_throughput,
            "time": elapsed,
        })

    # -------------------------------------------------------------------------
    # 7. Final Consolidated Summary Table
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("CALIBRATED CAPACITY BENCHMARK SUMMARY (WIKITEXT-2 / A100)")
    print("=" * 135)
    print(
        f"{'ARCHITECTURE':<36} | {'BACKBONE':<10} | {'TOTAL':<8} | {'TOKENS/PARAM':<14} | "
        f"{'BEST LOSS':<10} | {'FINAL PPL ↓':<12} | {'THROUGHPUT':<16}"
    )
    print("-" * 135)
    for r in summary_records:
        print(
            f"{r['name']:<36} | {r['backbone']:<10} | {r['total']:<8} | {r['ratio']:<14} | "
            f"{r['best_loss']:>8.4f}   | {r['final_ppl']:>10.2f}   | {r['throughput'] / 1e3:>6.1f}k tok/s"
        )
    print("=" * 135)


if __name__ == "__main__":
    run_benchmark()

[INFO] Device: NVIDIA A100-SXM4-80GB | Backend: gpu | Precision: bfloat16
[INFO] Dataset: WikiText-2 | Tokens: Train=2,448,382 | Val=258,659 | Vocab=50257
[INFO] Protocol: Steps=1200 | Batch=32 | Context=512 | D_Model=256

[INFO] Starting Architecture: Volterra-Fundamental (M=16)
       Total Parameters: 16.15M | Backbone: 3148.3k | Tokens/Param: 0.8:1
       XLA compilation completed in 24.44s
  Step  150/1200 | Val Loss: 5.1999 | PPL: 181.25 | Entropy: 4.416 | Kernel Max:  41.4 | Throughput: 535.6k tok/s
  Step  300/1200 | Val Loss: 4.8495 | PPL: 127.68 | Entropy: 4.082 | Kernel Max:  38.9 | Throughput: 525.7k tok/s
  Step  450/1200 | Val Loss: 4.6724 | PPL: 106.96 | Entropy: 3.975 | Kernel Max:  37.8 | Throughput: 522.9k tok/s
  Step  600/1200 | Val Loss: 4.5865 | PPL:  98.15 | Entropy: 3.913 | Kernel Max:  37.4 | Throughput: 521.0k tok/s
  Step  750/1200 | Val Loss: 4.5317 | PPL:  92.91 | Entropy: 3.892 | Kernel Max:  37.3 | Throughput: 519.3k tok/s
  Step  900/1200 | Val Loss: 4.5

In [ ]:
"""
run_volterra_pretrain_transfer.py

Calibrated Pre-training and Transfer Learning Benchmark for Volterra Attention
Dataset: WikiText-103 (Clean Wikipedia Prose) -> WikiText-2 (Target Domain)
Model Scale: Volterra-Micro (D=128, L=2, H=4, M=8 | Backbone: ~410k parameters)
Framework: JAX / Flax / Optax | Hardware: NVIDIA GPU (A100/L4) via XLA Native JIT
Precision: Native bfloat16 Mixed Precision
"""

import os
import math
import time
import zipfile
import urllib.request
from typing import Any, Dict, Tuple

import numpy as np

# Configure XLA memory allocation
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import jax
import jax.numpy as jnp
from jax import random
import flax.linen as nn
from flax.training import train_state
import optax

# -----------------------------------------------------------------------------
# 0. Global Setup & Calibrated Micro Hyperparameters
# -----------------------------------------------------------------------------
DEVICES = jax.devices()
print("=" * 135)
print(f"[INFO] Device: {DEVICES[0].device_kind.upper()} (Count: {len(DEVICES)}) | Backend: {jax.default_backend()}")
print(f"[INFO] Precision: bfloat16 Native via @jax.jit")
print("=" * 135)

BLOCK_SIZE: int = 512
BATCH_SIZE: int = 32
SEED: int = 42
DTYPE: jnp.dtype = jnp.bfloat16

# Calibrated Micro Architecture (Backbone ~410k params)
D_MODEL: int = 128
N_LAYERS: int = 2
N_HEADS: int = 4
M_MODES: int = 8

# Optimization Schedules
PRETRAIN_STEPS: int = 1500     # ~24.5M tokens seen during pre-training
PRETRAIN_LR: float = 8e-4

FINETUNE_STEPS: int = 800      # ~13.1M tokens during adaptation
FINETUNE_LR: float = 3e-4      # Moderate learning rate for positive transfer


# -----------------------------------------------------------------------------
# 1. Clean Dataset Ingestion (WikiText-103 & WikiText-2 via Direct Mirrors)
# -----------------------------------------------------------------------------
def load_datasets() -> Tuple[np.ndarray, np.ndarray, np.ndarray, int]:
    data_dir = "/tmp/clean_language_corpora"
    os.makedirs(data_dir, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0"}

    # 1.1 Target Domain: WikiText-2
    wt2_train_file = os.path.join(data_dir, "wt2_train.txt")
    wt2_valid_file = os.path.join(data_dir, "wt2_valid.txt")
    wt2_urls = {
        wt2_train_file: "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt",
        wt2_valid_file: "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/valid.txt"
    }

    print("[INFO] [1/2] Fetching WikiText-2 (Target Domain)...")
    for path, url in wt2_urls.items():
        if not os.path.exists(path) or os.path.getsize(path) < 1000:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=30) as resp, open(path, "wb") as f:
                f.write(resp.read())

    with open(wt2_train_file, "r", encoding="utf-8") as f: wt2_train_text = f.read()
    with open(wt2_valid_file, "r", encoding="utf-8") as f: wt2_valid_text = f.read()

    # 1.2 Pre-training Corpus: WikiText-103 (Clean Wikipedia Articles without XML)
    wt103_zip = os.path.join(data_dir, "wikitext-103-raw-v1.zip")
    wt103_raw = os.path.join(data_dir, "wikitext-103-raw", "wiki.train.raw")

    print("[INFO] [2/2] Fetching WikiText-103 Pre-training Corpus (Clean Prose, ~100M tokens)...")
    if not os.path.exists(wt103_raw) or os.path.getsize(wt103_raw) < 10_000_000:
        wt103_url = "https://wikitext.smerity.com/wikitext-103-raw-v1.zip"
        try:
            req = urllib.request.Request(wt103_url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as resp, open(wt103_zip, "wb") as f:
                f.write(resp.read())
            with zipfile.ZipFile(wt103_zip, "r") as z:
                z.extractall(data_dir)
            print("       WikiText-103 successfully extracted!")
        except Exception as e:
            print(f"       Primary S3 mirror failed ({e}), creating enlarged clean fallback...")
            with open(wt103_raw, "w", encoding="utf-8") as f:
                f.write((wt2_train_text + "\n") * 20)

    # Read up to the first 30 million characters (~6M-8M words) for fast, saturated tokenization
    with open(wt103_raw, "r", encoding="utf-8") as f:
        pretrain_text = f.read(35_000_000)

    # 1.3 Tokenization via GPT-2 BPE
    try:
        import tiktoken
        tokenizer = tiktoken.get_encoding("gpt2")
        pretrain_tokens = tokenizer.encode_ordinary(pretrain_text)
        wt2_train_tokens = tokenizer.encode_ordinary(wt2_train_text)
        wt2_valid_tokens = tokenizer.encode_ordinary(wt2_valid_text)
        vocab_size = tokenizer.n_vocab
    except ImportError:
        pretrain_tokens = list(pretrain_text.encode("utf-8"))
        wt2_train_tokens = list(wt2_train_text.encode("utf-8"))
        wt2_valid_tokens = list(wt2_valid_text.encode("utf-8"))
        vocab_size = 256

    pretrain_arr = np.array(pretrain_tokens, dtype=np.int32)
    wt2_tr_arr = np.array(wt2_train_tokens, dtype=np.int32)
    wt2_va_arr = np.array(wt2_valid_tokens, dtype=np.int32)

    print(f"[INFO] Pre-train Tokens (WikiText-103 subset): {len(pretrain_arr):,} tokens (~{len(pretrain_arr)/len(wt2_tr_arr):.1f}x WikiText-2)")
    print(f"[INFO] Target Domain (WikiText-2): Train={len(wt2_tr_arr):,} | Validation={len(wt2_va_arr):,} | Vocab={vocab_size}")
    return pretrain_arr, wt2_tr_arr, wt2_va_arr, vocab_size


PRETRAIN_DATA, WT2_TRAIN_DATA, WT2_VAL_DATA, VOCAB_SIZE = load_datasets()


def sample_batch(data_array: np.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
    max_idx = len(data_array) - BLOCK_SIZE - 1
    indices = np.random.randint(0, max_idx, size=(BATCH_SIZE,))
    x = np.stack([data_array[i : i + BLOCK_SIZE] for i in indices])
    y = np.stack([data_array[i + 1 : i + BLOCK_SIZE + 1] for i in indices])
    return jnp.array(x, dtype=jnp.int32), jnp.array(y, dtype=jnp.int32)


# -----------------------------------------------------------------------------
# 2. Geometric Hyperspherical Normalization
# -----------------------------------------------------------------------------
def project_conical(x: jnp.ndarray, eps: float = 1e-7) -> jnp.ndarray:
    scale = jnp.sqrt(float(x.shape[-1]))
    norm = jnp.linalg.norm(x.astype(jnp.float32), axis=-1, keepdims=True) + eps
    return (scale * (x.astype(jnp.float32) / norm)).astype(x.dtype)


class ConicalNorm(nn.Module):
    dim: int
    eps: float = 1e-7

    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        return project_conical(x, self.eps)


def identity_matrix_init(rng: Any, shape: Tuple[int, int], dtype: jnp.dtype = jnp.float32) -> jnp.ndarray:
    return jnp.eye(shape[0], shape[1], dtype=dtype)


# -----------------------------------------------------------------------------
# 3. Volterra Continuous Spectral Attention
# -----------------------------------------------------------------------------
class ScaledVolterraAttention(nn.Module):
    d_model: int = D_MODEL
    n_heads: int = N_HEADS
    m_modes: int = M_MODES
    block_size: int = BLOCK_SIZE
    dtype: jnp.dtype = DTYPE

    def setup(self) -> None:
        self.head_dim = self.d_model // self.n_heads

        t = jnp.arange(self.block_size, dtype=jnp.float32)
        m = jnp.arange(self.m_modes, dtype=jnp.float32)[None, :]
        t_normalized = (2.0 * t[:, None] / max(self.block_size - 1, 1)) - 1.0
        t_clipped = jnp.minimum(jnp.maximum(t_normalized, -1.0 + 1e-6), 1.0 - 1e-6)
        cheb = jnp.cos(m * jnp.arccos(t_clipped))

        x_s = (2.0 * t + 1.0) / (2.0 * self.block_size)
        w_cheb = 1.0 / jnp.sqrt(x_s * (1.0 - x_s) + 1e-5)
        w_cheb = (w_cheb / jnp.mean(w_cheb))[:, None]

        self.base_phi = cheb
        self.base_psi = cheb * w_cheb

        t_grid = t[:, None]
        s_grid = t[None, :]
        self.delta_ts = jnp.maximum(t_grid - s_grid, 0.0)
        self.causal_mask = jnp.tril(jnp.ones((self.block_size, self.block_size), dtype=jnp.bool_))

    @nn.compact
    def __call__(self, x: jnp.ndarray, audit: bool = False) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        b, t, c = x.shape

        c_attn = nn.Dense(3 * self.d_model, use_bias=False, dtype=self.dtype, name="c_attn")
        c_proj = nn.Dense(self.d_model, use_bias=False, dtype=self.dtype, name="c_proj")

        lambdas = self.param(
            "lambdas",
            lambda rng: jnp.ones((self.n_heads, self.m_modes), dtype=jnp.float32) / math.sqrt(self.m_modes)
        )
        init_gammas = jnp.broadcast_to(
            jnp.linspace(math.log(0.005), math.log(0.20), self.m_modes),
            (self.n_heads, self.m_modes)
        )
        log_gamma = self.param("log_gamma", lambda rng: init_gammas)

        adapt_phi = nn.Dense(self.m_modes, use_bias=False, kernel_init=identity_matrix_init, name="adapt_phi")
        adapt_psi = nn.Dense(self.m_modes, use_bias=False, kernel_init=identity_matrix_init, name="adapt_psi")

        qkv = c_attn(x)
        q, k, v = jnp.split(qkv, 3, axis=-1)

        q = q.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)
        k = k.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)
        v = v.reshape((b, t, self.n_heads, self.head_dim)).swapaxes(1, 2)

        q_proj = project_conical(q)
        k_proj = project_conical(k)

        phi = adapt_phi(self.base_phi[:t, :])
        psi = adapt_psi(self.base_psi[:t, :])

        gamma = jnp.minimum(jnp.maximum(jnp.exp(log_gamma), 1e-4), 2.0)
        gamma_exp = gamma[:, :, None, None]
        delta = self.delta_ts[:t, :t][None, None, :, :]

        decay_matrix = jnp.exp(-gamma_exp * delta)
        base_kernel = jnp.einsum("tm,sm->mst", phi, psi)[None, :, :, :]

        lam = lambdas[:, :, None, None]
        causal_kernel = jnp.sum(lam * base_kernel * decay_matrix, axis=1)
        causal_kernel = jnp.where(self.causal_mask[:t, :t][None, :, :], causal_kernel, 0.0)

        # Associative state integration: [1, H, T, T] @ [B, H, T, D_h]
        input_content = k_proj * v
        integrated_state = jnp.matmul(causal_kernel.astype(self.dtype)[None, :, :, :], input_content)
        state_conical = project_conical(integrated_state)

        out = q_proj * state_conical
        out = out.swapaxes(1, 2).reshape((b, t, c))

        if audit:
            max_val = jnp.max(jnp.abs(causal_kernel))
            k_abs = jnp.abs(causal_kernel)
            probs = k_abs / (jnp.sum(k_abs, axis=-1, keepdims=True) + 1e-7)
            probs = jnp.maximum(probs, 1e-9)
            entropy = -jnp.mean(jnp.sum(probs * jnp.log(probs), axis=-1))
        else:
            max_val = jnp.array(0.0, dtype=jnp.float32)
            entropy = jnp.array(0.0, dtype=jnp.float32)

        return c_proj(out), max_val, entropy


# -----------------------------------------------------------------------------
# 4. Volterra-Micro Model Architecture
# -----------------------------------------------------------------------------
class VolterraBlock(nn.Module):
    d_model: int = D_MODEL
    n_heads: int = N_HEADS
    m_modes: int = M_MODES

    @nn.compact
    def __call__(self, x: jnp.ndarray, audit: bool = False) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        norm1 = ConicalNorm(self.d_model, name="norm1")
        norm2 = ConicalNorm(self.d_model, name="norm2")
        attn = ScaledVolterraAttention(self.d_model, self.n_heads, self.m_modes, BLOCK_SIZE, DTYPE, name="attn")
        attn_out, max_val, entropy = attn(norm1(x), audit=audit)
        x = x + attn_out

        mlp_fc1 = nn.Dense(4 * self.d_model, use_bias=False, dtype=DTYPE, name="mlp_fc1")
        mlp_fc2 = nn.Dense(self.d_model, use_bias=False, dtype=DTYPE, name="mlp_fc2")
        x = x + mlp_fc2(nn.silu(mlp_fc1(norm2(x))))
        return x, max_val, entropy


class VolterraMicroLM(nn.Module):
    vocab_size: int = VOCAB_SIZE
    d_model: int = D_MODEL
    n_heads: int = N_HEADS
    n_layers: int = N_LAYERS
    m_modes: int = M_MODES

    @nn.compact
    def __call__(
        self,
        idx: jnp.ndarray,
        targets: jnp.ndarray = None,
        audit: bool = False
    ) -> Tuple[jnp.ndarray, Any, jnp.ndarray, jnp.ndarray]:
        b, t = idx.shape
        tok_embed = self.param("token_emb", nn.initializers.normal(stddev=0.02), (self.vocab_size, self.d_model), DTYPE)
        pos_embed = self.param("pos_emb", nn.initializers.normal(stddev=0.02), (BLOCK_SIZE, self.d_model), DTYPE)

        pos = jnp.arange(t)[None, :]
        x = tok_embed[idx] + pos_embed[pos]

        max_vals, entropies = [], []
        for i in range(self.n_layers):
            x, mv, ent = VolterraBlock(self.d_model, self.n_heads, self.m_modes, name=f"block_{i}")(x, audit=audit)
            if audit:
                max_vals.append(mv)
                entropies.append(ent)

        x = ConicalNorm(self.d_model, name="norm_final")(x)
        logits = jnp.matmul(x, tok_embed.T)

        loss = None
        if targets is not None:
            loss = optax.softmax_cross_entropy_with_integer_labels(
                logits=logits.astype(jnp.float32),
                labels=targets
            ).mean()

        if audit:
            peak_val = jnp.max(jnp.stack(max_vals))
            avg_ent = jnp.mean(jnp.stack(entropies))
        else:
            peak_val = jnp.array(0.0, dtype=jnp.float32)
            avg_ent = jnp.array(0.0, dtype=jnp.float32)

        return logits, loss, peak_val, avg_ent


# -----------------------------------------------------------------------------
# 5. Compiled Execution Functions
# -----------------------------------------------------------------------------
@jax.jit
def train_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[train_state.TrainState, jnp.ndarray]:
    def loss_fn(params):
        _, loss, _, _ = state.apply_fn({"params": params}, x, targets=y, audit=False)
        return loss

    grad_fn = jax.value_and_grad(loss_fn)
    loss, grads = grad_fn(state.params)
    return state.apply_gradients(grads=grads), loss


@jax.jit
def eval_step(state: train_state.TrainState, x: jnp.ndarray, y: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    _, loss, max_val, entropy = state.apply_fn({"params": state.params}, x, targets=y, audit=True)
    return loss, max_val, entropy


def evaluate_dataset(state: train_state.TrainState, data_array: np.ndarray, iters: int = 40) -> Tuple[float, float, float, float]:
    losses, max_vals, entropies = [], [], []
    for _ in range(iters):
        xb, yb = sample_batch(data_array)
        l, mv, ent = eval_step(state, xb, yb)
        losses.append(float(l))
        max_vals.append(float(mv))
        entropies.append(float(ent))
    v_loss = float(np.mean(losses))
    ppl = math.exp(min(v_loss, 15.0))
    return v_loss, ppl, float(np.max(max_vals)), float(np.mean(entropies))


# -----------------------------------------------------------------------------
# 6. Benchmark Pipeline Execution
# -----------------------------------------------------------------------------
def run_pipeline() -> None:
    rng = random.PRNGKey(SEED)
    rng, init_rng = random.split(rng)

    model = VolterraMicroLM(
        vocab_size=VOCAB_SIZE,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        m_modes=M_MODES
    )

    dummy_x = jnp.ones((BATCH_SIZE, BLOCK_SIZE), dtype=jnp.int32)
    params = model.init(init_rng, dummy_x, targets=dummy_x)["params"]

    total_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
    embed_params = VOCAB_SIZE * D_MODEL + BLOCK_SIZE * D_MODEL
    backbone_params = total_params - embed_params
    chinchilla_ratio = len(PRETRAIN_DATA) / max(backbone_params, 1)

    print("\n" + "=" * 135)
    print(f"CALIBRATED MODEL CAPACITY: VOLTERRA-MICRO")
    print(f"• Total Parameters: {total_params / 1e6:.2f}M | Backbone (Attention + MLPs): {backbone_params / 1e3:.1f}k")
    print(f"• Pre-train Tokens per Backbone Parameter: {chinchilla_ratio:.1f}:1 (Saturated Chinchilla Regime)")
    print("=" * 135)

    # -------------------------------------------------------------------------
    # STAGE 1: Pre-training on Clean WikiText-103
    # -------------------------------------------------------------------------
    print(f"\n[INFO] STAGE 1: Pre-training on WikiText-103 ({PRETRAIN_STEPS} Steps | LR={PRETRAIN_LR})")
    pretrain_schedule = optax.cosine_decay_schedule(init_value=PRETRAIN_LR, decay_steps=PRETRAIN_STEPS)
    tx_pretrain = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=pretrain_schedule, weight_decay=0.1, b1=0.9, b2=0.95)
    )
    state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx_pretrain)

    t_compile = time.time()
    dummy_xb, dummy_yb = sample_batch(PRETRAIN_DATA)
    state, _ = train_step(state, dummy_xb, dummy_yb)
    _ = eval_step(state, dummy_xb, dummy_yb)
    jax.block_until_ready(state.params)
    print(f"       XLA compilation completed in {time.time() - t_compile:.2f}s")

    t0 = time.time()
    for step in range(1, PRETRAIN_STEPS + 1):
        xb, yb = sample_batch(PRETRAIN_DATA)
        state, loss = train_step(state, xb, yb)

        if step % 300 == 0 or step == PRETRAIN_STEPS:
            jax.block_until_ready(loss)
            pt_loss, pt_ppl, _, _ = evaluate_dataset(state, PRETRAIN_DATA, iters=20)
            zs_loss, zs_ppl, _, _ = evaluate_dataset(state, WT2_VAL_DATA, iters=20)
            tok_s = (step * BATCH_SIZE * BLOCK_SIZE) / (time.time() - t0)
            print(
                f"  Step {step:>4}/{PRETRAIN_STEPS} | Pre-train Loss: {pt_loss:.4f} (PPL {pt_ppl:>5.1f}) | "
                f"WikiText-2 Zero-Shot PPL: {zs_ppl:>6.1f} | Throughput: {tok_s / 1e3:>5.1f}k tok/s"
            )

    # -------------------------------------------------------------------------
    # STAGE 2: Fine-Tuning on WikiText-2
    # -------------------------------------------------------------------------
    print(f"\n[INFO] STAGE 2: Fine-Tuning on WikiText-2 ({FINETUNE_STEPS} Steps | LR={FINETUNE_LR})")
    ft_schedule = optax.cosine_decay_schedule(init_value=FINETUNE_LR, decay_steps=FINETUNE_STEPS)
    tx_ft = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=ft_schedule, weight_decay=0.1, b1=0.9, b2=0.95)
    )
    state_ft = train_state.TrainState.create(apply_fn=model.apply, params=state.params, tx=tx_ft)

    t0 = time.time()
    for step in range(1, FINETUNE_STEPS + 1):
        xb, yb = sample_batch(WT2_TRAIN_DATA)
        state_ft, loss = train_step(state_ft, xb, yb)

        if step % 200 == 0 or step == FINETUNE_STEPS:
            jax.block_until_ready(loss)
            v_loss, v_ppl, _, _ = evaluate_dataset(state_ft, WT2_VAL_DATA, iters=30)
            print(f"  FT Step {step:>3}/{FINETUNE_STEPS} | WikiText-2 Val Loss: {v_loss:.4f} | PPL: {v_ppl:>6.2f}")

    ft_final_loss, ft_final_ppl, _, _ = evaluate_dataset(state_ft, WT2_VAL_DATA, iters=80)

    # -------------------------------------------------------------------------
    # STAGE 3: Counter-Baseline — Trained from Scratch on WikiText-2
    # -------------------------------------------------------------------------
    print(f"\n[INFO] STAGE 3: Baseline — Trained from Scratch on WikiText-2 ({FINETUNE_STEPS} Steps | Same LR)")
    rng, scratch_rng = random.split(rng)
    scratch_params = model.init(scratch_rng, dummy_x, targets=dummy_x)["params"]
    state_scratch = train_state.TrainState.create(apply_fn=model.apply, params=scratch_params, tx=tx_ft)

    for step in range(1, FINETUNE_STEPS + 1):
        xb, yb = sample_batch(WT2_TRAIN_DATA)
        state_scratch, loss = train_step(state_scratch, xb, yb)

        if step % 200 == 0 or step == FINETUNE_STEPS:
            jax.block_until_ready(loss)
            v_loss, v_ppl, _, _ = evaluate_dataset(state_scratch, WT2_VAL_DATA, iters=30)
            print(f"  Scratch Step {step:>3}/{FINETUNE_STEPS} | WikiText-2 Val Loss: {v_loss:.4f} | PPL: {v_ppl:>6.2f}")

    scratch_final_loss, scratch_final_ppl, _, _ = evaluate_dataset(state_scratch, WT2_VAL_DATA, iters=80)

    # -------------------------------------------------------------------------
    # 7. Final Consolidated Report
    # -------------------------------------------------------------------------
    print("\n" + "=" * 135)
    print("CONSOLIDATED TRANSFER LEARNING REPORT: WIKITEXT-103 -> WIKITEXT-2")
    print("=" * 135)
    print(f"{'LEARNING REGIME':<35} | {'PRE-TRAIN SOURCE':<24} | {'VAL LOSS WT-2 ↓':<16} | {'FINAL PPL WT-2 ↓'}")
    print("-" * 135)
    print(f"{'Trained from Scratch':<35} | {'None (WikiText-2 only)':<24} | {scratch_final_loss:>10.4f}       | {scratch_final_ppl:>12.2f}")
    print(f"{'Pre-trained + Fine-tuned':<35} | {'WikiText-103 (Clean)':<24} | {ft_final_loss:>10.4f}       | {ft_final_ppl:>12.2f}")
    print("=" * 135)

    delta_ppl = scratch_final_ppl - ft_final_ppl
    relative_gain = (delta_ppl / scratch_final_ppl) * 100.0
    if delta_ppl > 0:
        print(f"[POSITIVE TRANSFER CONFIRMED] Perplexity reduced by {delta_ppl:.2f} points ({relative_gain:.1f}% relative improvement) via pre-training.")
    else:
        print(f"[NOTE] Discrepancy observed: Scratch achieved lower perplexity by {-delta_ppl:.2f} points.")


if __name__ == "__main__":
    run_pipeline()

[INFO] Device: NVIDIA A100-SXM4-80GB (Count: 1) | Backend: gpu
[INFO] Precision: bfloat16 Native via @jax.jit
[INFO] [1/2] Fetching WikiText-2 (Target Domain)...
[INFO] [2/2] Fetching WikiText-103 Pre-training Corpus (Clean Prose, ~100M tokens)...
       WikiText-103 successfully extracted!
[INFO] Pre-train Tokens (WikiText-103 subset): 7,741,795 tokens (~3.2x WikiText-2)
[INFO] Target Domain (WikiText-2): Train=2,448,382 | Validation=258,659 | Vocab=50257

CALIBRATED MODEL CAPACITY: VOLTERRA-MICRO
• Total Parameters: 6.89M | Backbone (Attention + MLPs): 393.6k
• Pre-train Tokens per Backbone Parameter: 19.7:1 (Saturated Chinchilla Regime)

[INFO] STAGE 1: Pre-training on WikiText-103 (1500 Steps | LR=0.0008)
       XLA compilation completed in 21.63s
  Step  300/1500 | Pre-train Loss: 6.0494 (PPL 423.8) | WikiText-2 Zero-Shot PPL:  522.7 | Throughput: 716.0k tok/s
  Step  600/1500 | Pre-train Loss: 5.7029 (PPL 299.7) | WikiText-2 Zero-Shot PPL:  378.9 | Throughput: 692.9k tok/s
  Step